In [ ]:
import pandas as pd, numpy as np

# === 路徑 ===
a_path = 'ds_numService_with_billing_onehot.csv'   # A
b_path = 'final_customer_features_dataset.csv'     # B
c_path = 'cleaned_dataset2.csv'                    # C
out_path = 'csr_service_bill.csv'

# === A, B ===
a = pd.read_csv(a_path).rename(columns={'客編': 'CUST_NO'})
b = pd.read_csv(b_path)

# --- B 類別重組 ---
b['MAIN_CATEGORY'] = b['MAIN_CATEGORY'].where(
    b['MAIN_CATEGORY'].isin(b['MAIN_CATEGORY'].value_counts().nlargest(4).index),
    '其他主類')
b['SUB_CATEGORY'] = b['SUB_CATEGORY'].where(
    b['SUB_CATEGORY'].isin(b['SUB_CATEGORY'].value_counts().nlargest(6).index),
    '其他分類')
b_encoded = pd.get_dummies(b[['MAIN_CATEGORY','SUB_CATEGORY']],
                           prefix=['maincat','subcat'], dtype=np.int8)
b = pd.concat([b.drop(columns=['MAIN_CATEGORY','SUB_CATEGORY']), b_encoded], axis=1)

# --- C 使用狀態對照表 (EPON/CM) ---
cols = ['客編','產品名稱','用戶種類','相關編號','起日','迄日',
        '系統台','地區','繳別','使用狀態']
status_map = {}
for chunk in pd.read_csv(c_path, sep='^', engine='python',
                         names=cols, dtype=str, chunksize=300_000):
    sub = chunk[chunk['產品名稱'].isin(['EPON','CM'])]
    sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
    status_map.update(sub.groupby('客編')['status_val'].min().to_dict())

# --- 合併 A+B ---
merged = b.merge(a, on='CUST_NO', how='outer')

# --- 補齊使用狀態_數值 ---
mask = merged['使用狀態_數值'].isna()
merged.loc[mask, '使用狀態_數值'] = merged.loc[mask, 'CUST_NO'].map(status_map)

# --- 其餘空值補 0 ---
merged = merged.fillna(0)

# --- 輸出 ---
merged.to_csv(out_path, index=False)
print(f'Done → {out_path} , shape={merged.shape}')


C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

Done → csr_service_bill.csv , shape=(67249, 39)
